# Customer Coupon Acceptance Analysis
## Practical Application Assignment 5.1: Will the Customer Accept the Coupon?

This notebook analyzes driver survey data to determine key drivers of coupon acceptance, specifically investigating **Bar coupons** and **Coffee House coupons**.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os

# Load dataset from local path or fallback URL
data_path = 'data/coupons.csv' if os.path.exists('data/coupons.csv') else ('coupons.csv' if os.path.exists('coupons.csv') else 'https://raw.githubusercontent.com/ashwinkambli/coupon_acceptance/main/data/coupons.csv')
data = pd.read_csv(data_path)
data.head()

### Data Cleaning & Overview
- Inspect shape and missing values
- Handle columns with missing data

In [ ]:
# Display dataset shape and info
print('Dataset Shape:', data.shape)
missing_values = data.isnull().sum()
print('Missing Values:\n', missing_values[missing_values > 0])

# Handle missing values if any
data_clean = data.copy()

### Question 3: Overall Coupon & Bar Coupon Acceptance Rates
Calculate the overall proportion of accepted coupons across all types, and specifically for Bar coupons.

In [ ]:
# Overall coupon acceptance rate
overall_acceptance = data_clean['Y'].mean()
print(f'Overall Coupon Acceptance Rate: {overall_acceptance * 100:.2f}%')

# Filter strictly to Bar coupons
bar_coupons = data_clean[data_clean['coupon'] == 'Bar']
bar_acceptance = bar_coupons['Y'].mean()
print(f'Bar Coupon Acceptance Rate: {bar_acceptance * 100:.2f}% (Total: {len(bar_coupons)})')

### Question 4: Bar Visit Frequency Analysis
Compare the acceptance rate between drivers who visit a bar 3 or fewer times a month vs. more than 3 times a month (strictly within Bar coupon recipients).

In [ ]:
# Drivers visiting bars > 3 times a month
target_q4 = bar_coupons[bar_coupons['Bar'].isin(['4~8', 'gt8'])]
remaining_q4 = bar_coupons[~bar_coupons.index.isin(target_q4.index)]

rate_q4_target = target_q4['Y'].mean()
rate_q4_remaining = remaining_q4['Y'].mean()

print(f'Bar > 3 times/month acceptance rate: {rate_q4_target * 100:.2f}% ({target_q4["Y"].sum()}/{len(target_q4)})')
print(f'Remaining Bar coupon recipients acceptance rate: {rate_q4_remaining * 100:.2f}% ({remaining_q4["Y"].sum()}/{len(remaining_q4)})')

### Question 5: Age & Bar Visit Frequency Analysis
Compare acceptance rate between drivers who go to a bar more than once a month AND are over the age of 25 vs. remaining Bar coupon recipients.

In [ ]:
# Bar > 1 time/month and age > 25
bar_gt1 = bar_coupons['Bar'].isin(['1~3', '4~8', 'gt8'])
over_25 = ~bar_coupons['age'].isin(['21', 'below21'])

target_q5 = bar_coupons[bar_gt1 & over_25]
remaining_q5 = bar_coupons[~bar_coupons.index.isin(target_q5.index)]

rate_q5_target = target_q5['Y'].mean()
rate_q5_remaining = remaining_q5['Y'].mean()

print(f'Bar > 1/month & Age > 25 acceptance rate: {rate_q5_target * 100:.2f}% ({target_q5["Y"].sum()}/{len(target_q5)})')
print(f'Remaining Bar coupon recipients acceptance rate: {rate_q5_remaining * 100:.2f}% ({remaining_q5["Y"].sum()}/{len(remaining_q5)})')

### Question 6: Complex Target Group Analysis
Compare acceptance rates for specific target subgroups (using `passanger != 'Kid(s)'` for non-kid passengers) against remaining Bar coupon recipients.

In [ ]:
# 6A: Bar > 1/month, passenger != 'Kid(s)', occupation != 'Farming Fishing & Forestry'
non_kid = bar_coupons['passanger'] != 'Kid(s)'
not_farm = bar_coupons['occupation'] != 'Farming Fishing & Forestry'
target_6a = bar_coupons[bar_gt1 & non_kid & not_farm]
remaining_6a = bar_coupons[~bar_coupons.index.isin(target_6a.index)]

print(f'6A Acceptance Rate: {target_6a["Y"].mean() * 100:.2f}% vs Remaining: {remaining_6a["Y"].mean() * 100:.2f}%')

# 6B: Bar > 1/month, passenger != 'Kid(s)', maritalStatus != 'Widowed'
not_widowed = bar_coupons['maritalStatus'] != 'Widowed'
target_6b = bar_coupons[bar_gt1 & non_kid & not_widowed]
remaining_6b = bar_coupons[~bar_coupons.index.isin(target_6b.index)]

print(f'6B Acceptance Rate: {target_6b["Y"].mean() * 100:.2f}% vs Remaining: {remaining_6b["Y"].mean() * 100:.2f}%')

# 6C1: Bar > 1/month & age < 30
under_30 = bar_coupons['age'].isin(['below21', '21', '26'])
target_6c1 = bar_coupons[bar_gt1 & under_30]
remaining_6c1 = bar_coupons[~bar_coupons.index.isin(target_6c1.index)]

print(f'6C (Under 30) Acceptance Rate: {target_6c1["Y"].mean() * 100:.2f}% vs Remaining: {remaining_6c1["Y"].mean() * 100:.2f}%')

# 6C2: Bar > 1/month & cheap restaurants > 4/month & income < $50k
cheap_rest = bar_coupons['RestaurantLessThan20'].isin(['4~8', 'gt8'])
low_inc = bar_coupons['income'].isin(['Less than $12500', '$12500 - $24999', '$25000 - $37499', '$37500 - $49999'])
target_6c2 = bar_coupons[bar_gt1 & cheap_rest & low_inc]
remaining_6c2 = bar_coupons[~bar_coupons.index.isin(target_6c2.index)]

print(f'6C (Cheap Rest & Income < 50k) Acceptance Rate: {target_6c2["Y"].mean() * 100:.2f}% vs Remaining: {remaining_6c2["Y"].mean() * 100:.2f}%')

### Question 7: Formulated Hypotheses & Summary of Bar Coupon Findings

Based on the conditional acceptance probability analysis:
1. **Prior Behavior is Paramount:** Drivers who already frequent bars (> 1-3 times per month) accept Bar coupons at dramatically higher rates (~69-76%) compared to non-bar or infrequent bar visitors (~33-38%).
2. **Passenger Context Matters:** Having non-kid passengers (friends/partners) correlates with significantly higher coupon acceptance (70.18%) than traveling with kids or alone.
3. **Age Influence:** Drivers over 25 who visit bars regularly show strong acceptance (69.52%), while younger drivers (< 30) who visit bars regularly accept at an even higher rate (71.77%).

## Independent Investigation: Coffee House Coupons (`coupon == 'CoffeeHouse'`)

We investigate driver characteristics that lead to higher acceptance rates for **Coffee House** coupons.

In [ ]:
# Filter dataset strictly to Coffee House coupons
coffee_coupons = data_clean[data_clean['coupon'] == 'Coffee House']
coffee_acceptance = coffee_coupons['Y'].mean()
print(f'Overall Coffee House Acceptance Rate: {coffee_acceptance * 100:.2f}% ({coffee_coupons["Y"].sum()}/{len(coffee_coupons)})')

# 1. Acceptance rate by visit frequency
freq_coffee = coffee_coupons[coffee_coupons['CoffeeHouse'].isin(['1~3', '4~8', 'gt8'])]
infreq_coffee = coffee_coupons[coffee_coupons['CoffeeHouse'].isin(['never', 'less1'])]
print(f'Frequent Coffee House visitors (> 1/month): {freq_coffee["Y"].mean() * 100:.2f}%')
print(f'Infrequent Coffee House visitors (<= 1/month): {infreq_coffee["Y"].mean() * 100:.2f}%')

# 2. Acceptance rate by time of day
time_10am = coffee_coupons[coffee_coupons['time'] == '10AM']
time_2pm = coffee_coupons[coffee_coupons['time'] == '2PM']
time_midday = coffee_coupons[coffee_coupons['time'].isin(['10AM', '2PM'])]
time_evening = coffee_coupons[coffee_coupons['time'].isin(['6PM', '10PM'])]

print(f'10 AM Acceptance: {time_10am["Y"].mean() * 100:.2f}%')
print(f'2 PM Acceptance: {time_2pm["Y"].mean() * 100:.2f}%')
print(f'Combined 10AM & 2PM Acceptance: {time_midday["Y"].mean() * 100:.2f}%')
print(f'Evening (6PM & 10PM) Acceptance: {time_evening["Y"].mean() * 100:.2f}%')

# 3. Acceptance rate by passenger type
with_friends = coffee_coupons[coffee_coupons['passanger'] == 'Friend(s)']
alone = coffee_coupons[coffee_coupons['passanger'] == 'Alone']
print(f'Traveling with Friends Acceptance: {with_friends["Y"].mean() * 100:.2f}%')
print(f'Traveling Alone Acceptance: {alone["Y"].mean() * 100:.2f}%')

### Summary & Recommendations for Coffee House Coupons
1. **Target Frequent Visitors:** Drivers visiting Coffee Houses > 1/month accept at **65.93%** vs **38.14%** for infrequent visitors.
2. **Timing is Critical:** Morning (10 AM - 63.96%) and early afternoon (2 PM - 54.66%) achieve much higher acceptance than late evening (6 PM/10 PM - 35.29%).
3. **Social Context:** Drivers traveling with friends have a **59.69%** acceptance rate compared to **47.68%** when alone.